Lazy evaluation, planes de ejecución, Catalyst, SparkUI
Particiones, cache, broadcast joins, data skew
AQE, configuración, errores comunes, checklist producción

> **Dataset:** seguimos con NYC Yellow Taxi 2024-01 (~3 millones de filas, Parquet) — el mismo de la Clase 3 para que se enfoquen en las optimizaciones y no en aprender datos nuevos.


---
## Lazy evaluation, planes de ejecución y SparkUI

### Setup


In [1]:
import os
import sys
os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Best Practices Spark Local y SparkUI")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista. Versión:", spark.version)
print("SparkUI corre en:", spark.sparkContext.uiWebUrl)

SparkSession lista. Versión: 4.1.2
SparkUI corre en: http://127.0.0.1:4040


**Cargamos el dataset de NYC Taxi** (mismo de la Clase 3). Si lo tienen descargado, salten esta celda; si no, baja en ~30 segundos.

In [ ]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet -O /tmp/taxi.parquet
!ls -lh /tmp/taxi.parquet

-rw-r--r-- 1 root root 48M Mar 21  2024 /tmp/taxi.parquet


In [ ]:
df_taxi = spark.read.parquet("/tmp/taxi.parquet")
print(f"Filas: {df_taxi.count():,}")
df_taxi.printSchema()

Filas: 2,964,624
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



## 1.2 — Evaluación perezosa (lazy evaluation)

La idea central que hay que entender de Spark: **las transformaciones no ejecutan nada**. Solo describen "lo que vas a hacer". El motor recién corre el trabajo cuando llamas una **acción** (`count()`, `show()`, `collect()`, `write`...).

**Por qué importa:** Spark mira **todas** las transformaciones juntas antes de ejecutar y las **optimiza** como un solo plan. Si encadenas 10 filtros y selects, Spark puede combinarlos, reordenarlos y empujarlos hacia la fuente para leer menos datos.

Veámoslo en acción.


In [ ]:
from pyspark.sql.functions import col, year, month, hour, avg

# Definir transformaciones (NO SE EJECUTA NADA TODAVÍA)
df_pipeline = (
    df_taxi
    .filter(col("trip_distance") > 0)
    .filter(col("total_amount") > 0)
    .withColumn("anio",  year("tpep_pickup_datetime"))
    .withColumn("mes",   month("tpep_pickup_datetime"))
    .withColumn("hora",  hour("tpep_pickup_datetime"))
    .select("hora", "trip_distance", "total_amount")
)

# Hasta acá Spark NO ha tocado los datos
print("Tipo:", type(df_pipeline))
print("¿Es streaming?:", df_pipeline.isStreaming)
print("Schema (esto SÍ se calcula porque está en metadata):")
df_pipeline.printSchema()

Tipo: <class 'pyspark.sql.classic.dataframe.DataFrame'>
¿Es streaming?: False
Schema (esto SÍ se calcula porque está en metadata):
root
 |-- hora: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)



**Ahora, una acción.** Recién acá Spark dispara el cálculo real.

In [ ]:
# Esta línea es la que efectivamente lee el archivo y ejecuta el pipeline
import time
t0 = time.time()
n = df_pipeline.count()
print(f"Filas resultantes: {n:,} (cálculo en {time.time()-t0:.2f}s)")

## 1.3 — `explain()`: leer el plan de ejecución

`.explain()` te muestra **qué va a hacer Spark** antes de hacerlo. Es la herramienta #1 para optimizar.

Hay varios modos:
- `df.explain()` — solo el plan físico (lo que realmente va a ejecutarse).
- `df.explain(True)` — los 4 planes: parsed, analyzed, optimized, physical.
- `df.explain("formatted")` — versión más legible (Spark 3+).


In [ ]:
# Plan físico (corto)
df_pipeline.explain()

Ahora con detalle completo.

In [ ]:
df_pipeline.explain(True)

Spark primero muestra el plan “tal como lo escribiste”: filtros, columnas nuevas (`anio`, `mes`, `hora`) y selección final.
Luego lo optimiza: elimina columnas que no necesita y deja solo `hora`, `trip_distance` y `total_amount`.
Finalmente el plan físico dice cómo lo ejecutará: leerá solo esas 3 columnas del Parquet, filtrará valores mayores a 0 y calculará la hora.


**Lo importante que se ve en el plan optimizado:**

1. **`PushedFilters`**: los filtros `trip_distance > 0` y `total_amount > 0` fueron **empujados al lector de Parquet**. Spark lee solo lo que necesita.
2. **`ReadSchema`**: solo lee las columnas que vas a usar (no las 19 originales). Esto se llama **column pruning**.
3. **`Project`**: el `select` se combina con los filtros, no se ejecuta en una pasada separada.

Todo esto lo hace **Catalyst optimizer** automáticamente. No tuviste que pedirlo.


## 1.4 — Catalyst optimizer: reglas en acción

Catalyst aplica decenas de reglas. Las más útiles de conocer:

| Regla | Qué hace |
|-------|----------|
| **Predicate pushdown** | Empuja filtros hacia la fuente. Lee menos datos del disco. |
| **Column pruning** | Lee solo las columnas usadas. |
| **Constant folding** | `col + 0` se reescribe a `col`. |
| **Filter combining** | `filter(A).filter(B)` se vuelve `filter(A AND B)`. |
| **Join reordering** | Reordena joins para minimizar shuffle. |

Demostremos predicate pushdown con un ejemplo claro: leer solo viajes con monto > 50 USD.


In [ ]:
# Con filtro: Catalyst empuja el filtro al lector Parquet
df_caros = (
    spark.read.parquet("/tmp/taxi.parquet")
    .filter(col("total_amount") > 50)
    .select("tpep_pickup_datetime", "trip_distance", "total_amount")
)

df_caros.explain()
# Buscar: PushedFilters: [IsNotNull(total_amount), GreaterThan(total_amount,50.0)]

In [ ]:
# Mismo cálculo pero con el filtro escrito como una UDF (que Catalyst NO puede empujar)
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

@udf(BooleanType())
def es_caro(monto):
    return monto is not None and monto > 50

df_caros_udf = (
    spark.read.parquet("/tmp/taxi.parquet")
    .filter(es_caro(col("total_amount")))
    .select("tpep_pickup_datetime", "trip_distance", "total_amount")
)

df_caros_udf.explain()
# Aquí PushedFilters está VACÍO -> Spark lee todo el Parquet y filtra después.

**Comparemos los tiempos:**

In [ ]:
import time

t0 = time.time(); n1 = df_caros.count();     t1 = time.time() - t0
t0 = time.time(); n2 = df_caros_udf.count(); t2 = time.time() - t0

print(f"Con built-in (pushdown):  {n1:,} filas en {t1:.2f}s")
print(f"Con UDF (sin pushdown):   {n2:,} filas en {t2:.2f}s")
print(f"La UDF es {t2/t1:.1f}x más lenta")

**Lección:** evita UDFs cuando hay built-in. No es solo por la serialización Python↔JVM, también porque las UDFs **bloquean a Catalyst** (no puede empujar filtros, no puede reordenar joins, etc.).

## 1.5 — SparkUI: qué ver y cómo abrirlo

El SparkUI es el panel web donde Spark muestra **lo que pasó en cada job**. Tiene cuatro pestañas clave:

1. **Jobs**: cada acción dispara un job. Ves duración, etapas, tasks.
2. **Stages**: cada job se divide en stages separados por shuffles. Es donde detectas el cuello de botella.
3. **SQL / DataFrame**: visualiza los planes con tiempos.
4. **Executors**: memoria, GC, datos procesados por nodo.

**En Colab el UI corre en localhost:4040 (no accesible directamente)**. En clase nos vamos a apoyar en `.explain()` que es equivalente y siempre funciona. Si quieren ver el UI real:
- En su PC local: Spark expone http://localhost:4040 automáticamente.
- En Databricks / EMR / Synapse: viene integrado.
- En Colab: hay que tunelar con ngrok o Cloudflare (complicado, no lo veremos hoy).

Veamos qué pasó hasta ahora:


In [ ]:
print("URL del SparkUI (solo accesible si estuvieras local):", spark.sparkContext.uiWebUrl)
print()
print("Stages que se han ejecutado:")
print(spark.sparkContext.statusTracker().getJobIdsForGroup())
print()
print("Para inspeccionar planes en Colab, usaremos .explain() y la API de status.")

In [ ]:
print("SparkUI corre en:", spark.sparkContext.uiWebUrl)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

1. Jobs — qué se ejecutó y cuánto tardó
Lo que muestra: una fila por cada acción que ejecutaste (count, show, collect, write...). Para cada job: ID, descripción, hora, duración, cuántos stages tuvo, cuántos tasks.
Para qué sirve: ver el historial. "¿Cuál query me está tardando 30 segundos?" → la encontrás acá ordenada por duración.


In [ ]:
from pyspark.sql.functions import col
# Job 1
df_taxi.count()
# Job 2
df_taxi.filter(col("trip_distance") > 5).count()
# Job 3
df_taxi.select("PULocationID", "total_amount").show(5)

2. Stages — el corazón del debugging
Lo que muestra: cada job se parte en stages. Un stage es un bloque de trabajo que termina con un shuffle (groupBy, join, repartition, distinct). Por cada stage te muestra: duración, tasks que tuvo, shuffle read/write, GC time, memoria pico.
Para qué sirve: acá es donde encontrás los cuellos de botella reales. Si un stage tardó 50x lo que los demás, ahí está el problema. Si un task dentro de un stage tardó 100x los demás, tenés skew.

In [ ]:
from pyspark.sql.functions import avg, count
df_taxi.groupBy("PULocationID").agg(
    count("*").alias("viajes"),
    avg("total_amount").alias("ticket_promedio")
).show()

3. Storage — qué tenés cacheado
Lo que muestra: datasets que cacheaste con .cache() o .persist(). Para cada uno: nivel (memoria/disco/serializado), cuántas particiones, tamaño en memoria, tamaño en disco.
Para qué sirve: verificar que el cache se materializó (a veces uno hace .cache() y se olvida de llamar una acción que lo dispare), y ver cuánta memoria estás consumiendo.


In [ ]:
df_filtrado = df_taxi.filter(col("total_amount") > 30)
df_filtrado.cache()   # le dice a Spark: “guarda este resultado en memoria porque lo voy a reutilizar”.
df_filtrado.count()   # esta acción es la que materializa el cache

4. Environment — la configuración entera
Lo que muestra: TODAS las configs activas: Spark Properties, System Properties, classpath, hadoop properties, JARs. Es la versión completa de spark.conf.get(...).
Para qué sirve: chequear cosas antes de optimizar. "¿Está AQE prendida? ¿Cuánto es shuffle.partitions? ¿Qué versión de Java está corriendo?" Todo está acá.
No hace falta correr nada para verla — abre directo. Buscá en Spark Properties cosas como:

spark.sql.adaptive.enabled → si está en true, AQE activa
spark.sql.shuffle.partitions → 200 por default
spark.sql.autoBroadcastJoinThreshold → 10485760 (10 MB) por default

5. Executors — los workers
Lo que muestra: lista de executors (incluido el driver). En Colab vas a ver solo uno porque corre en modo local, pero igual te muestra: cores, memoria usada/disponible, tasks completados/fallidos, GC time, datos shuffleados.
Para qué sirve: en producción, detectar workers que están sufriendo (mucho GC, OOM, tasks fallidos). En Colab te sirve para ver cuánta memoria del driver estás usando.
Tampoco hace falta correr nada — siempre tiene una fila (driver) que va actualizándose.

6. SQL / DataFrame — aparece después de la primera query
Acá hay una sorpresa: tu screenshot tiene 5 pestañas, pero falta una sexta llamada "SQL / DataFrame". Aparece automáticamente apenas hagas una operación DataFrame o SQL. Probablemente ya te apareció después de los pasos anteriores.
Lo que muestra: diagrama gráfico del plan de ejecución, con cada nodo (Scan, Filter, HashAggregate, Exchange...) y el tiempo que tardó cada uno. Es la versión visual de .explain().
Para qué sirve: esta es probablemente la más útil de las 6 para optimizar. Click en una query reciente y te muestra el árbol completo. Buscás:

Exchange (= shuffle) en rojo o lento → cuello de botella
BroadcastHashJoin → join optimizado correctamente
Scan parquet con PushedFilters listados → predicate pushdown funcionando

Para que aparezca y se pueble:


In [ ]:
df_taxi.filter(col("total_amount") > 50).groupBy("PULocationID").count().show()

In [ ]:
from google.colab import output

# Abrir la pestaña Structured Streaming del SparkUI en una ventana aparte
print("Abriendo SparkUI → Structured Streaming...")
output.serve_kernel_port_as_window(4040, path='/StreamingQuery/')
print("Sugerencia: pon la ventana del SparkUI al costado del notebook para ver ambas.")

In [ ]:
from pyspark.sql.functions import col, window, count, avg

# Fuente sintética: 50 filas por segundo con timestamp y value
df_rate = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 50)
    .load()
)

# Agregación con ventana temporal para que el SparkUI muestre operaciones reales
df_agg = (
    df_rate
    .groupBy(window(col("timestamp"), "5 seconds"))
    .agg(
        count("*").alias("filas"),
        avg("value").alias("promedio")
    )
)

query = (
    df_agg.writeStream
    .format("memory")
    .queryName("demo_sparkui")
    .outputMode("complete")
    .start()
)

print(f"Stream iniciado. Nombre: {query.name}")
print(f"ID: {query.id}")
print()
print("VUELVE A LA PESTAÑA DE STRUCTURED STREAMING DEL SPARKUI Y ACTUALIZA.")
print("Podrás ver:")
print("  • Active Queries → 1 (con el nombre 'demo_sparkui')")
print("  • Input rows/second → ~50")
print("  • Processed rows/second → ~50 (Spark va al día)")
print("  • Batch Duration → cuánto tarda cada micro-batch")

In [ ]:
import time

print("Monitor en vivo del stream (también miren el SparkUI):\n")

for i in range(6):
    time.sleep(5)
    p = query.lastProgress
    if p is None:
        print(f"[{(i+1)*5:>3}s] (aún sin batches procesados)")
        continue
    print(f"[{(i+1)*5:>3}s] "
          f"batch={p['batchId']:>3} | "
          f"in={p.get('inputRowsPerSecond', 0):>5.1f}/s | "
          f"proc={p.get('processedRowsPerSecond', 0):>5.1f}/s")

# Detener el stream
query.stop()
print("\nStream detenido.")

# Mostrar las últimas ventanas calculadas
print("\nÚltimas 5 ventanas agregadas:")
spark.sql("""
    SELECT
        window.start AS inicio,
        window.end   AS fin,
        filas,
        ROUND(promedio, 2) AS promedio
    FROM demo_sparkui
    ORDER BY window.start DESC
    LIMIT 5
""").show(truncate=False)

1. Proxy nativo de Colab (sin instalar nada, sin cuenta)

---
# BLOQUE 2 — Particiones, cache y broadcast joins (60 min)

## 2.1 — Particiones: la unidad de paralelismo

Un DataFrame en Spark está dividido en **particiones**. Cada partición se procesa en un **task** independiente. **Cuántas particiones tengas determina cuánto paralelismo logras.**

- **Muy pocas particiones**: no aprovechas todos los cores. Subutilización.
- **Demasiadas particiones**: overhead de coordinación. Muchos archivitos chicos.

Las dos formas de cambiar particiones:

| Operación | Qué hace | Cuándo |
|-----------|----------|--------|
| `repartition(n)` | **Shuffle completo** para llegar a `n` particiones | Cuando necesitas **más** particiones o redistribución uniforme. |
| `coalesce(n)` | **Sin shuffle**, junta particiones existentes | Cuando necesitas **menos** particiones (típico antes de escribir). |


In [ ]:
print("Particiones actuales del df_taxi:", df_taxi.rdd.getNumPartitions())

In [ ]:
# Subir a 16 particiones (esto SÍ hace shuffle)
df_rep = df_taxi.repartition(16)
print("Después de repartition(16):", df_rep.rdd.getNumPartitions())

In [ ]:
# Bajar a 4 particiones (solo agrupa las existentes)
df_coa = df_taxi.coalesce(4)
print("Después de coalesce(4):", df_coa.rdd.getNumPartitions())

**¿Por qué `coalesce` no hace shuffle?**

Porque solo combina particiones que ya están juntas en el mismo nodo. Por eso es mucho más rápido, pero el **balance puede quedar feo** (una partición con muchos datos, otras chicas).

**Regla práctica:**
- Vas a escribir un dataset chico → `coalesce(1)` o `coalesce(2)` para tener pocos archivos de salida.
- Vas a hacer joins / agregaciones pesadas → `repartition(2 * num_cores)` para distribuir bien.
- Nunca uses `coalesce(1)` sobre datasets grandes: forzás a un solo task a procesar todo.

**`repartition` por columna** es útil cuando vas a hacer múltiples operaciones sobre la misma clave:


In [ ]:
# Repartition por columna: pone todas las filas del mismo passenger_count en la misma partición
df_part_col = df_taxi.repartition(8, "passenger_count")
print("Particiones:", df_part_col.rdd.getNumPartitions())
df_part_col.explain()

## 2.2 — `cache()` y `persist()`: ¿cuándo y cómo?

**Spark recomputa el DataFrame cada vez que llamas una acción**. Si vas a usar el mismo DataFrame en varios cálculos, conviene **guardarlo en memoria** para no recalcularlo.

```python
df.cache()              # guarda en memoria + disco (default: MEMORY_AND_DISK)
df.persist(StorageLevel.MEMORY_ONLY)   # solo memoria
df.persist(StorageLevel.DISK_ONLY)     # solo disco
df.unpersist()          # libera
```

**Regla:** solo cachea lo que vas a usar **más de una vez**. Cachear algo que se usa una sola vez es contraproducente (perdés tiempo en guardarlo y memoria).


In [ ]:
# Caso: vamos a usar df_pipeline en 3 cálculos distintos
from pyspark.sql.functions import sum as f_sum

# SIN CACHE: cada acción reescanea el archivo
t0 = time.time()
df_pipeline.count()
df_pipeline.agg(avg("total_amount")).collect()
df_pipeline.agg(f_sum("trip_distance")).collect()
print(f"Sin cache: {time.time()-t0:.2f}s")

In [ ]:
# CON CACHE: solo lee la primera vez
df_pipeline.cache()
df_pipeline.count()  # primera vez: lee + cachea

t0 = time.time()
df_pipeline.count()
df_pipeline.agg(avg("total_amount")).collect()
df_pipeline.agg(f_sum("trip_distance")).collect()
print(f"Con cache: {time.time()-t0:.2f}s")

df_pipeline.unpersist()  # liberar memoria cuando ya no se usa

**Storage levels útiles:**

| Level | Cuándo |
|-------|--------|
| `MEMORY_ONLY` | Datos chicos, querés máxima velocidad. Si no entra en memoria, se descarta. |
| `MEMORY_AND_DISK` (default de `cache()`) | Lo más usado. Memoria primero, disco como fallback. |
| `MEMORY_AND_DISK_SER` | Igual pero serializado (menos memoria, más CPU). |
| `DISK_ONLY` | Datos enormes que no entran en RAM. |

**Importante:** `.cache()` también es lazy. Recién se materializa cuando se ejecuta una acción.


## 2.3 — Broadcast joins: el truco que acelera 10x los joins

Cuando haces `A.join(B)`, Spark por default hace un **shuffle hash join** o **sort-merge join**: ambas tablas se redistribuyen por la clave entre todos los nodos. Esto es **caro** (mucho tráfico de red).

**Si B es chica** (digamos < 10 MB), conviene copiar B entera a cada nodo. A esto se le llama **broadcast join**: cada partición de A se cruza localmente con la copia local de B. **Cero shuffle, cero red.**

Spark hace broadcast automáticamente si detecta que B es chica (umbral por defecto: 10 MB). Pero a veces no lo detecta y conviene **forzarlo**:


In [ ]:
from pyspark.sql.functions import broadcast

# Imaginemos un catálogo chico de zonas (id, nombre)
from pyspark.sql import Row
zonas = spark.createDataFrame([
    Row(LocationID=1,   nombre="Newark Airport"),
    Row(LocationID=132, nombre="JFK Airport"),
    Row(LocationID=138, nombre="LaGuardia Airport"),
    Row(LocationID=161, nombre="Midtown Center"),
    Row(LocationID=237, nombre="Upper East Side South"),
])

# Join SIN broadcast explícito
df_join_normal = df_taxi.join(zonas, df_taxi.PULocationID == zonas.LocationID, "inner")
df_join_normal.explain()
# Mira el plan: vas a ver "BroadcastHashJoin" si Spark detectó la tabla chica.

In [ ]:
# Forzar broadcast explícito
df_join_bcast = df_taxi.join(broadcast(zonas), df_taxi.PULocationID == zonas.LocationID, "inner")
df_join_bcast.explain()
# Ahora SEGURO aparece BroadcastHashJoin

**Cuándo forzar broadcast:**
- Tabla < 100 MB (más allá es riesgoso, satura memoria del driver).
- Catálogos, dimensiones, lookups.
- Cuando Spark no lo detecta automáticamente (porque vino de un cálculo complejo).

**Configuración relacionada:**
```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)  # 50 MB
```


## 2.4 — Data skew: el enemigo silencioso

**Skew** = datos desbalanceados. Una clave tiene millones de filas, las otras tienen pocas. Cuando agrupas por esa clave, **un solo task tiene que procesar la mayor parte**, mientras los otros terminan rápido y quedan esperando.

**Ejemplo típico en taxis:** la mayoría de los viajes salen de Manhattan. Si agrupás por `PULocationID`, las zonas de Manhattan tendrán tareas gigantes y las del Bronx tareas chicas.

Veámoslo:


In [ ]:
# ¿Cuán desbalanceado es PULocationID?
top_zonas = (
    df_taxi.groupBy("PULocationID")
    .count()
    .orderBy(col("count").desc())
)
top_zonas.show(10)

Si la zona top tiene millones de viajes y la mediana son miles, hay skew. Esto se traduce en **un task que tarda 50x lo que tardan los demás**.

**Estrategias de mitigación:**

1. **AQE con skew join handling** (lo veremos en el bloque 3): Spark detecta y rompe las particiones skewed automáticamente. La mejor opción si tu versión lo soporta.
2. **Broadcast join** si la otra tabla del join es chica.
3. **Salting**: agregas una columna aleatoria a la clave para que las filas de la zona "gigante" se dispersen entre varias particiones. Después agregas dos veces (una con la sal, otra sin).

Demo de salting básico:


In [ ]:
from pyspark.sql.functions import lit, concat, rand, expr

# Antes (skew): groupBy directo
df_skew = (
    df_taxi.groupBy("PULocationID")
    .agg(avg("total_amount").alias("promedio"))
)
df_skew.explain()

In [ ]:
# Salting: agregamos una sal (entero aleatorio 0-9) a la clave
df_salted = (
    df_taxi
    .withColumn("sal", (rand() * 10).cast("int"))
    .groupBy("PULocationID", "sal")
    .agg(f_sum("total_amount").alias("monto_parcial"), col("PULocationID").alias("_keep"))
)

# Después agregamos de nuevo SIN la sal, sumando los parciales
from pyspark.sql.functions import sum as f_sum, avg as f_avg
df_final = (
    df_salted
    .groupBy("PULocationID")
    .agg(f_sum("monto_parcial").alias("monto_total"))
)

df_final.show(5)

**Cuándo vale la pena salting:**
- Cuando una sola clave domina (típicamente > 30% del dataset).
- Cuesta más código y CPU, pero distribuye el trabajo uniformemente.

**Cuándo NO hacer salting:**
- Si el skew es moderado (la clave dominante tiene 2x lo que las demás), no vale la pena.

**Recreo de 10 minutos.** Cuando volvamos: AQE, configuraciones de producción y checklist final.


---
# BLOQUE 3 — AQE, configuración y producción (60 min)

## 3.1 — AQE: Adaptive Query Execution

Hasta Spark 2.x, Spark elegía el plan de ejecución **antes** de empezar y no lo cambiaba. Si te equivocabas con `shuffle.partitions`, no podias hacer nada.

Spark 3+ trae **AQE**: el motor **mira las estadísticas reales en runtime** después de cada shuffle y **ajusta el plan dinámicamente**.

**Qué resuelve AQE:**

1. **Particiones automáticas**: si después del shuffle quedan particiones chicas, las junta. Si quedan particiones gigantes, las parte.
2. **Skew join handling**: detecta skew y rompe las particiones skewed.
3. **Switch a broadcast en runtime**: si al hacer el shuffle resulta que una tabla quedó chica, cambia a broadcast.

**Es la mejor cosa que le pasó a Spark en años.** Y está apagada por default hasta Spark 3.2. Desde 3.2 viene prendida.


In [ ]:
# Activamos AQE ahora
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Verificamos
print("AQE habilitado:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Skew join handling:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("Coalesce partitions:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))

In [ ]:
# Ahora corramos un agregado y veamos el plan
df_agg_aqe = (
    df_taxi
    .groupBy("PULocationID")
    .agg(avg("total_amount").alias("promedio"))
)

df_agg_aqe.explain()
# Notar el "AdaptiveSparkPlan" al inicio. Eso significa AQE está activo.

In [ ]:
t0 = time.time()
df_agg_aqe.count()
print(f"Con AQE: {time.time()-t0:.2f}s")

## 3.2 — `spark.sql.shuffle.partitions`: la config más importante

Esta config define cuántas particiones se generan después de un shuffle. El default es **200**.

- 200 está pensado para clusters grandes (decenas de nodos).
- En un Colab con 2 cores, 200 particiones = overhead absurdo.
- En un cluster de producción de 8 workers con 4 cores cada uno, 200 puede quedar chico.

**Regla práctica:** `2 * num_cores * num_workers`.

Ejemplos:
- Colab 2 cores → 4 particiones.
- Cluster 8 workers × 4 cores → 64 particiones.
- Cluster 100 workers × 8 cores → 1600.

Cuando AQE está activo, Spark ajusta esto en runtime, así que ya no es tan crítico. Pero sigue siendo el techo desde el cual empieza a ajustar.


In [ ]:
# Sin AQE: si pones 200 particiones, vas a tener 200 archivos al escribir
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "false")

print("Particiones después de un groupBy:")
df_taxi.groupBy("PULocationID").count().rdd.getNumPartitions()

In [ ]:
# Con AQE: Spark ajusta automáticamente a algo más razonable
spark.conf.set("spark.sql.adaptive.enabled", "true")

print("Particiones después del MISMO groupBy con AQE:")
df_taxi.groupBy("PULocationID").count().rdd.getNumPartitions()

## 3.3 — Errores comunes y cómo debuggear

**1. `OutOfMemoryError: Java heap space`**

- Causa típica: `collect()` sobre un DataFrame gigante. `collect()` trae todos los datos al driver.
- Solución: usar `take(n)` o `show(n)` en vez de `collect()`. Si necesitás todos los datos, escribelos a disco.

**2. Job se cuelga indefinidamente**

- Causa típica: data skew (un task con todos los datos).
- Solución: activar AQE con skew handling. Como último recurso, salting.

**3. `org.apache.spark.SparkException: Job aborted`**

- Causa típica: alguna fila con datos malformados (null donde no debería, tipo incompatible).
- Solución: revisar el stacktrace, ver qué stage falló, qué partición. Filtrar nulls antes.

**4. Lectura lenta de archivos**

- Causa: muchos archivos chicos (small files problem) o CSV en vez de Parquet.
- Solución: convertir a Parquet con tamaño objetivo de 128-256 MB por archivo.

**5. Joins lentos**

- Causa: ambas tablas son grandes y se hace sort-merge join.
- Solución: si una es chica → `broadcast(df)`. Si ambas son grandes → revisar si AQE puede ayudar, sino particionar bien.

**6. `Task xxx is at risk of being killed because GC overhead`**

- Causa: GC excesivo, normalmente por estructuras de datos enormes en cada task.
- Solución: aumentar memoria del executor, bajar el tamaño de las particiones (más particiones, cada una más chica).


## 3.4 — Checklist de buenas prácticas

Antes de mandar un job a producción, repaso:

### Código
- [ ] **Sin UDFs si hay built-in equivalente**. Si no hay, intenté `pandas_udf` antes de UDF normal.
- [ ] **Schema explícito** en todas las lecturas de CSV / JSON.
- [ ] **Cache solo lo que se usa más de una vez**. Y siempre con `unpersist()` al final.
- [ ] **Sin `collect()` en datasets grandes**. Usar `show`, `take`, o escribir a disco.

### Optimización
- [ ] **`.explain()` de los queries críticos**. Buscar `PushedFilters` y `BroadcastHashJoin` donde corresponda.
- [ ] **`broadcast(small_df)`** explícito en joins con catálogos.
- [ ] **`shuffle.partitions` ajustado** a la infraestructura.
- [ ] **AQE habilitado** (por default desde Spark 3.2).

### Escritura
- [ ] **Parquet** como formato de salida (no CSV salvo para humanos).
- [ ] **`partitionBy()`** por columnas que se filtran (típicamente fecha).
- [ ] **Tamaño objetivo de archivo: 128-256 MB**. Si vas a quedar con muchos archivos chicos, `coalesce` antes.
- [ ] **`checkpointLocation`** si es streaming.

### Operación
- [ ] **Logging** de filas leídas vs escritas en cada paso.
- [ ] **Idempotencia**: si el job se re-corre, el resultado es el mismo.
- [ ] **Métricas** de duración y memoria por stage.


## 3.5 — Caso integrador: optimizar un pipeline real

Tenemos un pipeline de ejemplo (intencionalmente mal escrito). Vamos a aplicarle todo lo aprendido para acelerarlo.


In [ ]:
# Pipeline NO optimizado (intencional)
def pipeline_malo():
    df = spark.read.parquet("/tmp/taxi.parquet")

    # Anti-patrón: filtrar después de varias transformaciones inútiles
    df = df.withColumn("dummy", lit(1))
    df = df.withColumn("anio",  year("tpep_pickup_datetime"))
    df = df.withColumn("mes",   month("tpep_pickup_datetime"))

    # Cuello: joinear sin broadcast con una tabla chica
    zonas_local = spark.createDataFrame([
        Row(LocationID=i, nombre=f"Zona_{i}") for i in range(1, 250)
    ])
    df = df.join(zonas_local, df.PULocationID == zonas_local.LocationID)

    # Y un filtro tardío
    df = df.filter(col("total_amount") > 30)
    return df.count()

t0 = time.time()
res_malo = pipeline_malo()
t_malo = time.time() - t0
print(f"Pipeline malo: {res_malo:,} filas en {t_malo:.2f}s")

In [ ]:
# Pipeline OPTIMIZADO
def pipeline_bueno():
    df = (
        spark.read.parquet("/tmp/taxi.parquet")
        .filter(col("total_amount") > 30)              # 1. Filter early (pushdown)
        .select("tpep_pickup_datetime", "PULocationID", "total_amount", "trip_distance")  # 2. Column pruning
    )

    zonas_local = spark.createDataFrame([
        Row(LocationID=i, nombre=f"Zona_{i}") for i in range(1, 250)
    ])

    # 3. Broadcast join explícito
    df = df.join(broadcast(zonas_local), df.PULocationID == zonas_local.LocationID)
    return df.count()

t0 = time.time()
res_bueno = pipeline_bueno()
t_bueno = time.time() - t0
print(f"Pipeline bueno: {res_bueno:,} filas en {t_bueno:.2f}s")
print(f"Mejora: {t_malo/t_bueno:.1f}x más rápido")

## 3.6 — Cierre y puente a la Clase 6

**Lo que se llevan hoy:**

1. **Spark es perezoso**. Las transformaciones no ejecutan nada hasta una acción.
2. **`.explain()` es tu mejor amigo**. Buscar `PushedFilters`, `BroadcastHashJoin`, `AdaptiveSparkPlan`.
3. **Catalyst optimiza automáticamente**, pero **las UDFs lo bloquean**. Built-in siempre que se pueda.
4. **`repartition` vs `coalesce`**: el primero hace shuffle, el segundo no.
5. **Cache solo lo que se reutiliza**. Y siempre `unpersist()`.
6. **`broadcast(df)` para tablas chicas en joins**: 10x más rápido.
7. **Data skew**: AQE primero, salting como último recurso.
8. **AQE habilitado por default** desde Spark 3.2. Apagarla solo si tenés razón.
9. **`shuffle.partitions = 2 * cores`** como punto de partida.
10. **Checklist antes de producción**: schema explícito, sin collect, broadcast, particionado.

**Próxima clase (Sesión 6, última del módulo):** MLlib. Pipeline de Machine Learning sobre Spark. Vamos a usar el dataset MovieLens (que ya conocen de la tarea) para construir un **sistema de recomendación con ALS**.

**Tarea opcional:**
- Tomen su solución de la tarea práctica y aplíquenle `.explain()`. ¿Qué filtros se pushdown? ¿Hay broadcast joins?
- Comparen tiempos de su pipeline con vs sin AQE.


In [ ]:
# Cerrar SparkSession
spark.stop()
print("SparkSession cerrada")